# Historical espresso exploration
Read-only analysis of the authoritative `data/historical_shots_corrected.csv` (refreshed 2026-09-14). Run all cells after installing `.[dev,analysis]`.
The typed calculations live in `espresso_dialin.historical`; this notebook presents their results.

Settings are exact categories in first-observed order, not a calibrated fineness scale. Contiguous bean labels are conservative analysis blocks, **not verified sessions**. No missing measurement is inferred. Approximate corrected puck doses retain that interpretation; their uncertainty is not numerically invented.

Shots 37–39 are 3D, as confirmed by the user on 2026-09-14; the source records this correction.


In [ ]:
import hashlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from espresso_dialin.historical import (
    describe,
    eligible,
    exclusion_reasons,
    load_shots,
    metric,
    research_report,
    rolling_dose_report,
)

ROOT = Path.cwd()
if not (ROOT / "data/historical_shots_corrected.csv").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data/historical_shots_corrected.csv"
shots = load_shots(DATA)
report = research_report(shots)
print("Python", sys.version)
print("CSV SHA256", hashlib.sha256(DATA.read_bytes()).hexdigest())

## Audit and explicit eligibility
Output-rate analysis needs positive duration and grinder output; categorical output comparisons also need a setting and bean label. Yield-pair analysis only needs positive brew time and final yield. Primary extraction additionally requires exact setting, bean label and a usable dose interpretation (`TO_TARGET`, `MEASURED`, or consistent `NONE`). `UNKNOWN` is excluded even if the puck column says 18 g. `partial` is not a global exclusion. The approximate output on shot 40 is retained with a sensitivity check below.

Chronological changes require immediate original neighbours with consecutive sequence numbers, known settings and the same block. Missing rows/settings are never bridged. The extraction rolling comparison may use earlier known shots at the same exact setting within the block; it makes no adjacency claim.


In [ ]:
audit = report["audit"]
display(pd.Series({"rows": audit["rows"], **audit["statuses"]}, name="count"))
display(pd.DataFrame(audit["eligibility_by_bean"]).T)
display(pd.Series(audit["missingness"], name="blank_cells").to_frame())
exclusions = [
    {
        "analysis": analysis,
        "sequence": shot.sequence,
        "status": shot.raw["transcription_status"],
        "reasons": ", ".join(reasons),
    }
    for analysis in ("output", "setting_output", "yield_pair", "extraction")
    for shot in shots
    if (reasons := exclusion_reasons(shot, analysis))
]
display(pd.DataFrame(exclusions))
print({k: audit[k] for k in ("known_transitions", "dose_pairs", "rolling_extraction")})

## A. Grinder output and noise
Each panel keeps a bean block separate. Colour denotes an exact setting, without implying distance or direction. Duration was adjusted by the operator, so a pooled slope would be confounded with setting, chronology and bean changes. Compare g/s distributions and identical-duration repeats rather than fitting a physical scale.


In [ ]:
output = eligible(shots, "output")
output_frame = pd.DataFrame(
    [
        {
            "sequence": s.sequence,
            "bean": s.bean,
            "block": s.block,
            "setting": s.setting or "unknown",
            "duration_s": metric(s, "grind_duration_s"),
            "output_g": metric(s, "grinder_output_g"),
            "rate_g_s": metric(s, "rate"),
            "status": s.raw["transcription_status"],
        }
        for s in output
    ]
)
fig, axes = plt.subplots(1, 2, figsize=(13, 4), layout="constrained")
for ax, (bean, frame) in zip(axes, output_frame.groupby("bean", sort=False), strict=True):
    for setting, group in frame.groupby("setting", sort=False):
        ax.scatter(group.duration_s, group.output_g, label=setting)
    ax.set(title=bean, xlabel="Grind duration (s)", ylabel="Original grinder output (g)")
    ax.legend(ncol=3, fontsize=8)
plt.show()
display(pd.DataFrame(report["output_groups"]))
display(
    output_frame.groupby(["bean", "setting", "duration_s"], sort=False)
    .output_g.agg(["count", "mean", "median", "std", "min", "max"])
    .query("count >= 2")
)
display(
    pd.DataFrame(
        [
            {"bean": bean, **describe([metric(s, "rate") for s in output if s.bean == bean])}
            for bean in dict.fromkeys(s.bean for s in shots)
        ]
    )
)

## B. Extraction: raw time, actual yield, and the T36 approximation
The three outcomes share exactly the same extraction-eligible shots. Points retain all observations; black bars are medians. T36 is a crude derived value, not an observed target time. Repeated-setting groups also match block and puck-dose interpretation/target. Identical labels across bean groups must not be treated as replicates.


In [ ]:
extraction = eligible(shots, "extraction")
fig, axes = plt.subplots(2, 3, figsize=(14, 8), layout="constrained")
for row, bean in enumerate(dict.fromkeys(s.bean for s in shots)):
    members = [s for s in extraction if s.bean == bean]
    settings = list(dict.fromkeys(s.setting for s in members))
    for ax, field, label in zip(
        axes[row],
        ("brew_duration_s", "final_yield_g", "t36"),
        ("Raw brew time (s)", "Final yield (g)", "Linear T36 (s)"),
        strict=True,
    ):
        for index, setting in enumerate(settings):
            values = [metric(s, field) for s in members if s.setting == setting]
            ax.scatter([index] * len(values), values, alpha=0.7)
            ax.plot([index - 0.2, index + 0.2], [describe(values)["median"]] * 2, color="black")
        ax.set(xticks=range(len(settings)), xticklabels=settings, title=bean, ylabel=label)
plt.show()
display(pd.DataFrame(report["extraction_groups"]))
display(pd.DataFrame(report["variability"]))

## C. Robust description, not retrospective bad-shot diagnosis
Modified absolute z-score = `0.67448975 * abs(x - median) / MAD`; flag above 3.5 only in exact extraction groups with at least five observations. Zero MAD is explicitly unscorable. Group sizes below five are too sparse for this rule. No flag removes a shot, and full-group retrospective scores never enter rolling predictions.

Café Intención 3E (13 shots) and 3D (five shots) qualify; neither produces flags for either time representation. The 24/48 s pair at earlier 3F and the 38/62 s pair at newer 4E are large repeat differences, but two observations cannot identify which shot is anomalous. In particular the 17 s shot at newer 5H is a singleton, not a known channeling event. Means/SDs and medians/MADs above show sensitivity without inventing bad-shot labels.


In [ ]:
flags = pd.DataFrame(report["flags"])
display(flags)
print("Scorable shots per metric:", flags.groupby("metric").size().to_dict())
print("Flagged:", flags.loc[flags.flag.eq(True), "sequence"].tolist())

## D. Chronology and possible retention
The timeline displays missing extraction values as gaps. Changed/unchanged comparisons are descriptive and heavily confounded by the chosen setting. The strongest available simple comparison is a first changed shot and its immediate repeat at the same setting and puck interpretation. Previous settings are categorical; no signed or linear grind-distance feature is invented.


In [ ]:
chronology = pd.DataFrame(report["chronology"])
display(chronology)
display(
    chronology.groupby(["bean", "setting", "changed"], sort=False).t36.agg(
        ["count", "mean", "median", "std"]
    )
)
followups = pd.DataFrame(report["change_followups"])
display(followups)
fig, axes = plt.subplots(2, 1, figsize=(13, 6), layout="constrained")
for ax, bean in zip(axes, dict.fromkeys(s.bean for s in shots), strict=True):
    members = [s for s in shots if s.bean == bean]
    ax.plot(
        [s.sequence for s in members],
        [
            s.t36_linear_approx_s if not exclusion_reasons(s, "extraction") else float("nan")
            for s in members
        ],
        marker="o",
    )
    changed = chronology.loc[(chronology.bean == bean) & chronology.changed]
    ax.scatter(
        changed.sequence, changed.t36, marker="x", s=90, color="red", label="First after change"
    )
    ax.set(title=bean, xlabel="Original sequence", ylabel="Linear T36 (s)")
    ax.legend()
plt.show()

There are 43 extraction shots with known previous settings (18 changed, 25 unchanged), but still only seven first/repeat pairs (six Café Intención, one REWE). Repeat-minus-first T36 goes in both directions. Setting 50 is now known, so 50 → 51 is a known transition; missing yield on 50 still prevents an extraction repeat pair. The gap between REWE 4E shots 43 and 51 is not an immediate retention replicate. There is insufficient evidence for previous-setting predictive value or a latent retention model.


## E. Minimal chronological baselines
**Dose:** from the previous compatible shot, `t_new = t_old * 18 / output_old`. Score the implied previous output rate at the next shot's **actual** grind duration. Compatibility requires immediate same-setting neighbours in the same block, with both duration/output pairs known. Compare with simply carrying forward the last measured output. Report recommendations alongside observed durations; do not pretend the operator followed them.

**Extraction:** expanding past-only median at the same exact setting/block/puck target, minimum one prior observation. Score both baselines against the same observed raw brew duration. For the normalized method, multiply the past median T36 by the held-out **observed yield / 36**. This is conditional reconstruction using an outcome-side yield, not a deployable next-shot predictor, a measurement of true T36 error, or a causal experiment. All observations are retained; no future-derived outlier filter is applied.

**Existing rolling dose controllers:** compare last-shot proportional and past-only median rate on the same eligible observations. Both may revisit earlier exact-setting history within the bean block; they need not use immediate neighbours. These 33 predictions are a separate sample from the 29 adjacent pairs. Score output at actual duration, never at the unobserved recommended duration.


In [ ]:
display(pd.DataFrame(report["dose_pairs"]))
display(pd.DataFrame(report["rolling_extraction"]))
display(pd.DataFrame(report["errors"]))
rolling_dose = rolling_dose_report(shots)
display(pd.DataFrame(rolling_dose["rows"]))
display(pd.DataFrame(rolling_dose["overall"]).T)
print("Rolling dose by block:", rolling_dose["by_block"])

## Sensitivity to explicit uncertainty
Shot 40 has approximate grinder output; excluding it changes the descriptive rate distribution but none of the compatible dose pairs. Only row 14 retains UNKNOWN correction (row 28 is now TO_TARGET): list its derived T36, then show what repeated-setting SDs would look like **if** the entered 18 g were accepted as a comparable dose. This is a separate sensitivity, never a modification of the loaded rows or the primary eligibility rules.


In [ ]:
display(
    pd.DataFrame(
        [
            {
                "bean": bean,
                "exclude_approximate": exclude,
                **describe(
                    [
                        metric(s, "rate")
                        for s in output
                        if s.bean == bean
                        and (not exclude or s.raw["transcription_status"] != "approximate")
                    ]
                ),
            }
            for bean in dict.fromkeys(s.bean for s in shots)
            for exclude in (False, True)
        ]
    )
)
uncertain = [s for s in shots if exclusion_reasons(s, "extraction") == ("usable_puck_dose",)]
display(
    pd.DataFrame(
        [
            {
                "sequence": s.sequence,
                "mode": s.raw["dose_correction_mode"],
                "recorded_puck_g": s.numbers["puck_dose_g"],
                "t36": s.t36_linear_approx_s,
            }
            for s in uncertain
        ]
    )
)
sensitivity = pd.DataFrame(
    [
        {
            "bean": s.bean,
            "block": s.block,
            "setting": s.setting,
            "puck": s.numbers["puck_dose_g"],
            "raw": metric(s, "brew_duration_s"),
            "t36": metric(s, "t36"),
        }
        for s in shots
        if s in extraction or s in uncertain
    ]
)
display(
    sensitivity.groupby(["bean", "block", "setting", "puck"], sort=False)[["raw", "t36"]].agg(
        ["count", "median", "std"]
    )
)

## Conclusions and smallest next step
- Corrected data provide 50 output-rate observations, 45 primary extraction observations, 29 adjacent dose pairs and 28 past-only extraction comparisons. Known beans are Café Intención (1–39) and REWE Bio Espresso (40–51); sessions remain unknown.
- Preserve actual yield: shot 9 is 28 s at 43 g, giving a crude T36 of 23.44 s, not measured target time.
- Normalization remains inconsistent: pooled within-setting SD changes 5.891 → 6.138 s for Café Intención (33 shots, seven groups), and 12.500 → 11.544 s for REWE (four shots, two groups). Conditional extraction MAE changes 5.692 → 5.915 s (26 comparisons) and 15.500 → 14.201 s (two). These use held-out observed yield and are not prospective normalized forecasts.
- Adjacent proportional dose prediction has 0.711 g MAE versus 0.694 g for carrying output forward (29 pairs); only 12 pairs change duration. Proportionality is a baseline assumption, not a demonstrated improvement.
- The already-implemented rolling median-rate baseline has 0.675 g MAE versus 0.761 g for last-shot proportional on 33 matched predictions. Its advantage comes from Café Intención; the three REWE predictions are identical. This supports keeping the comparator, not claiming controller savings.
- The MAD screen finds no flags in Café Intención 3E (13 shots) and 3D (five shots). Variability at identical settings/durations remains substantial; absence of flags does not establish absence of bad preparation.
- Seven immediate change/repeat pairs still leave retention unresolved. Known settings add adjacency but do not recover missing yield or session/purge context.

**Next step:** prospective data acquisition using the existing dose baselines, with recommendations saved before outcomes. Prioritize controlled duration variation and replicated transitions where historical observations cannot separate effects; REWE has no identical-setting/duration output replicates. Do not repeat historical conditions without a specific remaining question. Defer extraction/retention complexity until it demonstrates out-of-sample value.

See `docs/historical-analysis.md` for current findings and experiment gaps. Coffee-to-target, causal savings and uncertainty calibration remain unmeasured.
